In [3]:
import json
import pandas as pd
from pathlib import Path

In [4]:
with open("diverse_summ.json", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame([
    {"eid": e["eid"], "aid": a["aid"], "title": a["title"], "url": a["url"],
     "domain": a["domain"], "content_part_1": a["content"][:32767],
     "content_part_2": a["content"][32767:65534]}
    for e in data for a in e["articles"]
])
df.to_excel("diverse_summ.xlsx", index=False)
print(f"Saved {len(df)} articles.")

Saved 2450 articles.


In [ ]:

# File locations
json_file = Path("diverse_summ.json")
output_file = json_file.with_name("diverse_summ_articles.xlsx")

# Read JSON
with open(json_file, "r", encoding="utf-8") as file:
    data = json.load(file)

rows = []

# Extract every article, including articles without question-answer references
for event in data:
    original_eid = event.get("eid", "")

    for article in event.get("articles", []):
        rows.append({
            "original_eid": original_eid,
            "original_aid": article.get("aid", ""),
            "Reference Article Title": article.get("title", ""),
            "Website": article.get("url", "")
        })

df = pd.DataFrame(rows)

# Remove completely duplicated article rows
df = df.drop_duplicates().reset_index(drop=True)


def create_sequential_ids(column):
    """Give identical original IDs the same sequential number."""
    mapping = {}
    result = []
    next_id = 1

    for value in column:
        if pd.isna(value) or str(value).strip() == "":
            result.append(pd.NA)
            continue

        value = str(value).strip()

        if value not in mapping:
            mapping[value] = next_id
            next_id += 1

        result.append(mapping[value])

    return pd.Series(result, dtype="Int64")


# Create independent sequential IDs
df["Event ID"] = create_sequential_ids(df["original_eid"])

# Assign each article a sequential number within its event
df["article_number"] = (
    df.groupby("Event ID", sort=False)["original_aid"]
      .transform(lambda column: pd.factorize(column, sort=False)[0] + 1)
)

# Combine Event ID and article number
# Example: Event 1 → 1.1, 1.2; Event 2 → 2.1, 2.2
df["Reference Article ID"] = (
    df["Event ID"].astype(str)
    + "."
    + df["article_number"].astype(str)
)

# Keep only the required columns
final_df = df[
    [
        "Event ID",
        "Reference Article ID",
        "Reference Article Title",
        "Website"
    ]
]

# Save to Excel
final_df.to_excel(
    output_file,
    index=False,
    sheet_name="Reference Articles"
)

print(f"File saved to: {output_file}")
print(f"Number of article rows: {len(final_df)}")
print(f"Unique events: {final_df['Event ID'].nunique()}")
print(
    "Unique articles:",
    final_df["Reference Article ID"].nunique()
)

# Confirm that the specific article was included
search_text = "desantis-trump-indictment-stormy-daniels"

matching_article = final_df[
    final_df["Website"].str.contains(
        search_text,
        case=False,
        na=False
    )
]

print("\nMatching article:")
print(matching_article.to_string(index=False))

File saved to: diverse_summ_articles.xlsx
Number of article rows: 2447
Unique events: 245
Unique articles: 2447

Matching article:
 Event ID Reference Article ID                                                                                Reference Article Title                                                                                                              Website
        1                  1.3 Ron DeSantis breaks silence on Trump indictment with quiet dig at 'alleged affair' with Stormy Daniels https://www.independent.co.uk/news/world/americas/us-politics/desantis-trump-indictment-stormy-daniels-b2304433.html


In [6]:
# check missing values
print(df.isnull().sum())

eid                0
article_aid        0
article_title      0
article_url        0
article_domain     0
article_content    0
question           0
answer             0
answer_aid         0
dtype: int64


In [10]:
#make data readable
input_file = Path("diverse_summ.xlsx")
output_file = input_file.with_name(input_file.stem + "_extracted.xlsx")
#required attributes
EID_COLUMN = "eid"
AID_COLUMN = "article_aid"
TITLE_COLUMN = "article_title"
WEBSITE_COLUMN = "article_url"
# assign sequential numbers to EID and AID columns
def create_sequential_ids(column):
    id_mapping = {}
    sequential_values = []
    next_id = 1
    for value in column:
        if pd.isna(value) or str(value).strip() == "":
            sequential_values.append(pd.NA)
            continue
        original_id = str(value).strip()
        if original_id not in id_mapping:
            id_mapping[original_id] = next_id
            next_id += 1
        sequential_values.append(id_mapping[original_id])
    return pd.Series(sequential_values, dtype="Int64")
# Create independent sequential IDs
df[EID_COLUMN] = create_sequential_ids(df[EID_COLUMN])
# AID numbering continues throughout the entire dataset
df[AID_COLUMN] = create_sequential_ids(df[AID_COLUMN])
# Keep only the required four columns
final_df = df[[
        EID_COLUMN,
        AID_COLUMN,
        TITLE_COLUMN,
        WEBSITE_COLUMN
    ]].copy()
# Rename columns for the output
final_df.columns = [
    "Event ID",
    "Reference Article ID",
    "Reference Article Title",
    "Website"
]
# Save the result
final_df.to_excel(output_file,index=False,sheet_name="Reference Articles")
# Remove completely duplicated rows
final_df = final_df.drop_duplicates().reset_index(drop=True)
# Save the cleaned data
final_df.to_excel(
    output_file,
    index=False,
    sheet_name="Reference Articles"
)
print(f"File saved to: {output_file}")
print(f"Rows after removing duplicates: {len(final_df)}")
print(f"Unique Event IDs: {final_df['Event ID'].nunique()}")
print(
    "Unique Reference Article IDs:",
    final_df["Reference Article ID"].nunique()
)


File saved to: diverse_summ_extracted.xlsx
Rows after removing duplicates: 1839
Unique Event IDs: 245
Unique Reference Article IDs: 1743
